In [1]:
import pandas as pd

In [2]:
df1 = pd.read_csv(r"C:\Users\reinh\Downloads\Projects\Online Retail II UCI\online_retail_II.csv", encoding="ISO-8859-1")
df1["InvoiceDate"] = pd.to_datetime(df1["InvoiceDate"])
print(df1.columns.tolist())
print(df1.shape)

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
(1067371, 8)


In [3]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [4]:
df2 = pd.read_csv(r"C:\Users\reinh\Downloads\Projects\Telecom Customer Churn\WA_Fn-UseC_-Telco-Customer-Churn.csv", encoding="ISO-8859-1")
print(df2.columns.tolist())
print(df2.shape)

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
(7043, 21)


In [5]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [6]:
import pandas as pd
from pathlib import Path

RAW = r"C:\Users\reinh\Downloads\Projects\Online Retail II UCI\online_retail_II.csv"  # adjust to your path
OUT = Path(r"C:\Users\reinh\Downloads\Projects/online_retail_clean.csv")

df = pd.read_csv(RAW, encoding="ISO-8859-1", parse_dates=["InvoiceDate"])
print(f"Loaded:                       {len(df):>10,}")

# 1. Drop rows with no Customer ID
before = len(df)
df = df[df["Customer ID"].notna()]
print(f"1. No Customer ID removed:    {before - len(df):>10,}  -> {len(df):,} left")

# 2. Remove cancellations (Invoice starts with 'C')
before = len(df)
df = df[~df["Invoice"].astype(str).str.startswith("C")]
print(f"2. Cancellations removed:     {before - len(df):>10,}  -> {len(df):,} left")

# 3. Remove zero or negative Quantity or Price
before = len(df)
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
print(f"3. Qty/Price <= 0 removed:    {before - len(df):>10,}  -> {len(df):,} left")

# 4. Customer ID to integer
df["Customer ID"] = df["Customer ID"].astype(int)
print("4. Customer ID converted to int")

# 5. Save
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print(f"5. Saved {len(df):,} rows to {OUT}")

Loaded:                        1,067,371
1. No Customer ID removed:       243,007  -> 824,364 left
2. Cancellations removed:         18,744  -> 805,620 left
3. Qty/Price <= 0 removed:            71  -> 805,549 left
4. Customer ID converted to int
5. Saved 805,549 rows to C:\Users\reinh\Downloads\Projects\online_retail_clean.csv


In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

CLEAN = r"C:\Users\reinh\Downloads\Projects\online_retail_clean.csv"
OUT   = r"C:\Users\reinh\Downloads\Projects\rfm_table.csv"
K     = 4  # change after checking the silhouette scores below

df = pd.read_csv(CLEAN, parse_dates=["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["Price"]
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)

# Raw RFM per customer
last_purchase = df.groupby("Customer ID")["InvoiceDate"].max()
rfm = pd.DataFrame({
    "Recency":   (snapshot - last_purchase).dt.days,
    "Frequency": df.groupby("Customer ID")["Invoice"].nunique(),
    "Monetary":  df.groupby("Customer ID")["Revenue"].sum(),
})
print(f"Customers: {len(rfm):,}")
print(rfm.describe().round(1))

# Quintile scores (rank first avoids duplicate-edge errors in qcut)
rfm["R_score"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_score"] = rfm[["R_score", "F_score", "M_score"]].sum(axis=1)

# K-Means on log-transformed, scaled RFM
X = StandardScaler().fit_transform(np.log1p(rfm[["Recency", "Frequency", "Monetary"]]))

print("\nSilhouette scores (higher is better):")
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    print(f"  k={k}: {silhouette_score(X, labels, sample_size=5000, random_state=42):.3f}")

km = KMeans(n_clusters=K, n_init=10, random_state=42)
rfm["Cluster"] = km.fit_predict(X)

print(f"\nCluster profile (k={K}):")
print(rfm.groupby("Cluster").agg(
    Customers=("Recency", "size"),
    Recency=("Recency", "mean"),
    Frequency=("Frequency", "mean"),
    Monetary=("Monetary", "mean"),
).round(1))

#rfm.to_csv(OUT)
print(f"\nSaved {len(rfm):,} customers to {OUT}")

Customers: 5,878
       Recency  Frequency  Monetary
count   5878.0     5878.0    5878.0
mean     201.3        6.3    3018.6
std      209.3       13.0   14737.7
min        1.0        1.0       3.0
25%       26.0        1.0     348.8
50%       96.0        3.0     898.9
75%      380.0        7.0    2307.1
max      739.0      398.0  608821.6

Silhouette scores (higher is better):


C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=23.
  warnings.warn(


  k=2: 0.441


C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=23.
  warnings.warn(


  k=3: 0.351


C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=23.
  warnings.warn(


... (7 outputs left out)

C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=23.
  warnings.warn(


  k=8: 0.295


C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=23.
  warnings.warn(



Cluster profile (k=4):
         Customers  Recency  Frequency  Monetary
Cluster                                         
0             1975    395.5        1.4     325.9
1             1250     28.3        3.0     867.0
2             1191     27.5       19.3   10998.3
3             1462    228.6        5.1    1995.3

Saved 5,878 customers to C:\Users\reinh\Downloads\Projects\rfm_table.csv


In [13]:
import pandas as pd
import numpy as np
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data, calibration_and_holdout_data

CLEAN = r"C:\Users\reinh\Downloads\Projects\online_retail_clean.csv"
RFM   = r"C:\Users\reinh\Downloads\Projects\rfm_table.csv"
OUT   = r"C:\Users\reinh\Downloads\Projects\clv_table.csv"

df = pd.read_csv(CLEAN, parse_dates=["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["Price"]

# One row per invoice, so frequency counts orders rather than line items
inv = df.groupby(["Customer ID", "Invoice", "InvoiceDate"], as_index=False)["Revenue"].sum()

# Weekly units; 10% annual discount rate converted to weekly
WEEKLY_DISCOUNT = 1.10 ** (1 / 52) - 1

# --- 1. Customer summary: frequency, recency, age, avg order value ---
summary = summary_data_from_transaction_data(
    inv, "Customer ID", "InvoiceDate",
    monetary_value_col="Revenue", freq="W",
)
print("Customers:", len(summary))
print(summary.describe().round(2))

# --- 2. Purchase model (BG/NBD) ---
bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(summary["frequency"], summary["recency"], summary["T"])
print("\nBG/NBD parameters:\n", bgf.summary.round(3))

# --- 3. Spend model (Gamma-Gamma), fit on repeat buyers only ---
repeat = summary[summary["frequency"] > 0]
ggf = GammaGammaFitter(penalizer_coef=0.0)
ggf.fit(repeat["frequency"], repeat["monetary_value"])
print("\nGamma-Gamma parameters:\n", ggf.summary.round(3))

# One-time buyers have no repeat spend, so give them the average repeat order value
summary.loc[summary["frequency"] == 0, "monetary_value"] = repeat["monetary_value"].mean()

# --- 4. Predict 12 months ahead ---
summary["predicted_orders_12m"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    52, summary["frequency"], summary["recency"], summary["T"]
)
summary["expected_order_value"] = ggf.conditional_expected_average_profit(
    summary["frequency"], summary["monetary_value"]
)
summary["CLV_12m"] = summary["predicted_orders_12m"] * summary["expected_order_value"]

print("\nSanity check:")
print(f"  Avg predicted orders (12m): {summary['predicted_orders_12m'].mean():.2f}")
print(f"  Avg expected order value:   {summary['expected_order_value'].mean():.2f}")
print(f"  Total 12m value:            {summary['CLV_12m'].sum():,.0f}")

# --- 5. Holdout check: fit on data before the last 90 days, test on those 90 days ---
cutoff = inv["InvoiceDate"].max() - pd.Timedelta(days=90)
ch = calibration_and_holdout_data(
    inv, "Customer ID", "InvoiceDate",
    calibration_period_end=cutoff,
    observation_period_end=inv["InvoiceDate"].max(),
    freq="W", monetary_value_col="Revenue",
)
bgf_cal = BetaGeoFitter(penalizer_coef=0.0).fit(
    ch["frequency_cal"], ch["recency_cal"], ch["T_cal"]
)
ch["predicted_holdout"] = bgf_cal.predict(
    ch["duration_holdout"], ch["frequency_cal"], ch["recency_cal"], ch["T_cal"]
)
print(f"\nHoldout check (last 90 days):")
print(f"  Actual repeat orders:    {ch['frequency_holdout'].sum():,.0f}")
print(f"  Predicted repeat orders: {ch['predicted_holdout'].sum():,.0f}")
print(f"  Correlation (customer level): {ch[['frequency_holdout','predicted_holdout']].corr().iloc[0,1]:.3f}")

# --- 6. Join with RFM clusters and summarize ---
rfm = pd.read_csv(RFM)
clv = summary.reset_index().merge(rfm[["Customer ID", "Cluster"]], on="Customer ID", how="left")
#clv.to_csv(OUT, index=False)

print("\nCLV (12 months, GBP) by RFM cluster:")
print(clv.groupby("Cluster").agg(
    Customers=("CLV_12m", "size"),
    Avg_CLV=("CLV_12m", "mean"),
    Total_CLV=("CLV_12m", "sum"),
).round(1))
print(f"\nTotal predicted 12-month value: {clv['CLV_12m'].sum():,.0f}")
print("\nTop 10 customers by CLV:")
print(clv.nlargest(10, "CLV_12m")[["Customer ID", "frequency", "recency", "monetary_value", "CLV_12m"]].round(1))

print(f"\nSaved to {OUT}")

Customers: 5878
       frequency  recency        T  monetary_value
count    5878.00  5878.00  5878.00         5878.00
mean        4.34    39.06    67.55          328.39
std         7.63    37.00    31.87         2269.61
min         0.00     0.00     0.00            0.00
25%         0.00     0.00    45.00            0.00
50%         2.00    31.50    76.00          216.96
75%         5.00    73.00    95.00          379.94
max       101.00   105.00   105.00       168469.60

BG/NBD parameters:
          coef  se(coef)  lower 95% bound  upper 95% bound
r       0.711     0.017            0.676            0.745
alpha  10.397     0.321            9.769           11.026
a       0.100     0.016            0.069            0.130
b       2.483     0.596            1.316            3.651

Gamma-Gamma parameters:
       coef  se(coef)  lower 95% bound  upper 95% bound
p    2.275     0.116            2.047            2.503
q    3.452     0.109            3.239            3.665
v  452.811    35.403   

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    GridSearchCV, cross_val_score, cross_val_predict,
)
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

CLEAN = r"C:\Users\reinh\Downloads\Projects\online_retail_clean.csv"
OUT   = r"C:\Users\reinh\Downloads\Projects\uci_customer_scores.csv"

df = pd.read_csv(CLEAN, parse_dates=["InvoiceDate"])
df["Revenue"] = df["Quantity"] * df["Price"]

END = df["InvoiceDate"].max()
CUTOFF = END - pd.Timedelta(days=90)
print(f"Features use data up to {CUTOFF.date()}; outcome is {CUTOFF.date()} to {END.date()}")

# 1. Features from the past only
past = df[df["InvoiceDate"] <= CUTOFF]
orders = past.groupby(["Customer ID", "Invoice", "InvoiceDate"], as_index=False)["Revenue"].sum()
feat = orders.groupby("Customer ID").agg(
    last=("InvoiceDate", "max"),
    first=("InvoiceDate", "min"),
    frequency=("Invoice", "count"),
    monetary=("Revenue", "sum"),
)
feat["recency"] = (CUTOFF - feat["last"]).dt.days
feat["tenure"] = (CUTOFF - feat["first"]).dt.days
feat["avg_order"] = feat["monetary"] / feat["frequency"]

# 2. Outcome from the next 90 days
future_spend = df[df["InvoiceDate"] > CUTOFF].groupby("Customer ID")["Revenue"].sum()
feat["future_spend"] = future_spend.reindex(feat.index).fillna(0)
feat["churned"] = (feat["future_spend"] == 0).astype(int)
print(f"\nCustomers at cutoff: {len(feat):,}")
print(f"Churn rate (no purchase in next 90 days): {feat['churned'].mean():.1%}")

# 3. Segments rebuilt from past-only features, then checked against the outcome
X_rfm = StandardScaler().fit_transform(np.log1p(feat[["recency", "frequency", "monetary"]]))
feat["segment"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(X_rfm)

profile = feat.groupby("segment").agg(
    customers=("recency", "size"),
    median_recency=("recency", "median"),
    median_frequency=("frequency", "median"),
    median_monetary=("monetary", "median"),
    churn_rate=("churned", "mean"),
    mean_future_spend=("future_spend", "mean"),
).round(2)
print("\nSegment outcomes (sorted from most to least recent):")
print(profile.sort_values("median_recency"))

# 4. Churn model: tuned with CV, threshold chosen on training data only
features = ["recency", "frequency", "monetary", "tenure", "avg_order"]
X = feat[features]
y = feat["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

candidates = {
    "Logistic regression": (
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        {"logisticregression__C": [0.01, 0.1, 1, 10]},
    ),
    "Gradient boosting": (
        GradientBoostingClassifier(random_state=42),
        {"n_estimators": [100, 300], "max_depth": [2, 3], "learning_rate": [0.05, 0.1]},
    ),
}

results = {}
for name, (est, grid) in candidates.items():
    gs = GridSearchCV(est, grid, scoring="roc_auc", cv=cv, n_jobs=-1).fit(X_tr, y_tr)
    best = gs.best_estimator_
    rep = cross_val_score(best, X_tr, y_tr, scoring="roc_auc", cv=rcv, n_jobs=-1)

    oof = cross_val_predict(best, X_tr, y_tr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
    thresholds = np.linspace(0.1, 0.9, 81)
    t_best = thresholds[int(np.argmax([f1_score(y_tr, oof >= t) for t in thresholds]))]

    proba = best.predict_proba(X_te)[:, 1]
    pred = (proba >= t_best).astype(int)
    results[name] = (best, t_best, rep.mean())

    print(f"\n=== {name} ===")
    print(f"Best params: {gs.best_params_}")
    print(f"Repeated CV ROC-AUC (25 folds): {rep.mean():.3f} ± {rep.std():.3f}")
    print(f"Threshold chosen on training CV: {t_best:.2f}")
    print(f"Test ROC-AUC: {roc_auc_score(y_te, proba):.3f}")
    print(f"Test churn recall: {recall_score(y_te, pred):.2f}, precision: {precision_score(y_te, pred):.2f}")

# 5. Score every customer with the better model, using out-of-fold predictions
winner = max(results, key=lambda n: results[n][2])
best, t, _ = results[winner]
print(f"\nBest model by repeated CV: {winner}")
feat["churn_probability"] = cross_val_predict(best, X, y, cv=cv, method="predict_proba")[:, 1]
feat["at_risk_flag"] = (feat["churn_probability"] >= t).astype(int)
feat.reset_index().to_csv(OUT, index=False)
print(f"Saved customer scores to {OUT}")

Features use data up to 2011-09-10; outcome is 2011-09-10 to 2011-12-09

Customers at cutoff: 5,281
Churn rate (no purchase in next 90 days): 56.6%


C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=21.
  warnings.warn(



Segment outcomes (sorted from most to least recent):
         customers  median_recency  median_frequency  median_monetary  \
segment                                                                 
2              893            19.0              14.0          5246.63   
0              661            26.0               2.0           605.55   
3             1606           166.0               5.0          1502.28   
1             2121           327.0               1.0           301.54   

         churn_rate  mean_future_spend  
segment                                 
2              0.16            2121.38  
0              0.44             412.02  
3              0.53             334.30  
1              0.80             163.83  

=== Logistic regression ===
Best params: {'logisticregression__C': 10}
Repeated CV ROC-AUC (25 folds): 0.784 ± 0.014
Threshold chosen on training CV: 0.38
Test ROC-AUC: 0.798
Test churn recall: 0.92, precision: 0.68

=== Gradient boosting ===
Best params: {'le

In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\reinh\Downloads\Projects\uci_customer_scores_v2.csv")
names = {2: "Champions", 0: "Recent", 3: "At Risk", 1: "Lost"}
df["Segment"] = df["segment"].map(names)
print(df.groupby("Segment").agg(
    Customers=("Customer ID", "size"),
    Avg_churn=("churn_probability", "mean"),
    Avg_CLV=("CLV_12m_cutoff", "mean"),
    Total_CLV=("CLV_12m_cutoff", "sum"),
    Revenue_at_risk=("value_at_risk", "sum"),
).round(2))

           Customers  Avg_churn  Avg_CLV   Total_CLV  Revenue_at_risk
Segment                                                              
At Risk         1606       0.53   861.29  1383235.97        626124.35
Champions        893       0.17  5425.14  4844646.03        585723.06
Lost            2121       0.80   282.12   598378.91        457152.49
Recent           661       0.45   960.48   634874.78        265734.80
